# ルールでラベルを付けて分類器を学習する（弱教師あり学習、Snorkel）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/weak_supervision.ipynb)

NLP クックブック 12-19 ／ テキスト分類

人手でラベルを付ける代わりに、キーワードや既存のモデルを使った「ラベル付け関数」を何本か書いてラベルを作り、そのラベルで分類器を学習したい。livedoor ニュースコーパスのうち 4 カテゴリ（スポーツ・映画・スマートフォン・独身女性の生活）を使う。

In [1]:
!pip install -q snorkel

In [2]:
import io, re, tarfile, urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# livedoor ニュースコーパス (株式会社ロンウイット配布、CC BY-ND 2.1 JP: 改変禁止)。記事は読み込んで使うだけで、改変したものを配布しない
CATS = ["sports-watch", "movie-enter", "smax", "dokujo-tsushin"]
ABSTAIN, SPORTS, MOVIE, SMAX, DOKUJO = -1, 0, 1, 2, 3   # -1 は「判断しない (棄権)」

url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
with urllib.request.urlopen(url) as r:
    tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
rows = []
for m in tar.getmembers():
    parts = m.name.split("/")
    if m.isfile() and len(parts) == 3 and parts[1] in CATS and parts[2] != "LICENSE.txt":
        lines = tar.extractfile(m).read().decode("utf-8").split("\n")   # 1 行目 URL, 2 行目 日付, 3 行目 タイトル
        rows.append({"y": CATS.index(parts[1]), "title": lines[2], "text": lines[2] + "\n" + "\n".join(lines[3:])[:300]})
df = pd.DataFrame(rows)
# train の正解ラベルは「使わない」前提。正解はラベル付け関数の評価と比較のためだけに見る
train, test = train_test_split(df, test_size=0.3, stratify=df["y"], random_state=0)
train, test = train.reset_index(drop=True), test.reset_index(drop=True)
print(len(train), len(test))

2457 1053


In [3]:
from sentence_transformers import SentenceTransformer
from snorkel.labeling import LFAnalysis, PandasLFApplier, labeling_function

def keyword_lf(name, pattern, label):
    """正規表現に当たればそのラベル、当たらなければ棄権するラベル付け関数を作る"""
    rx = re.compile(pattern)
    @labeling_function(name=name)
    def lf(x):
        return label if rx.search(x.text) else ABSTAIN
    return lf

lfs = [
    keyword_lf("sports_kw", r"試合|選手|五輪|サッカー|野球|ゴール|優勝", SPORTS),
    keyword_lf("coach", r"監督", SPORTS),                    # わざと入れた質の悪い関数 (映画の「監督」にも当たる)
    keyword_lf("movie_kw", r"映画|上映|公開|主演|劇場", MOVIE),
    keyword_lf("smax_kw", r"Android|スマホ|スマートフォン|ドコモ|NTT|au|ソフトバンク|アプリ", SMAX),   # au は "beautiful" などの英単語の一部にも当たる
    keyword_lf("dokujo_kw", r"独女|結婚|彼氏|婚活|アラサー|女性", DOKUJO),   # 「独女」は配信元のサイト名 (独女通信) なので、ほぼ答えの漏れ
    keyword_lf("title_bracket", r"^[^\n]*『", MOVIE),        # タイトルに『』があれば作品名とみなす
]

# 既存のモデルもラベル付け関数にできる: タイトルとラベルの説明文の埋め込みの類似度 (ゼロショット)。1 位と 2 位の差が小さいときは棄権
embedder = SentenceTransformer("intfloat/multilingual-e5-small")   # MIT ライセンス
desc = ["野球やサッカーなどスポーツの試合や選手に関する話題", "映画の公開、俳優、監督など映画に関する話題",
        "スマートフォンや携帯電話、モバイル通信に関する話題", "独身女性の恋愛、結婚、仕事、生活に関する話題"]
desc_emb = embedder.encode(["query: " + d for d in desc], normalize_embeddings=True)
for d in (train, test):   # 3,500 件のタイトルの埋め込み。Colab の CPU で 5〜8 分
    sims = embedder.encode(["query: " + t for t in d["title"]], normalize_embeddings=True, batch_size=64) @ desc_emb.T
    top2 = np.sort(sims, axis=1)[:, -2:]
    d["zero_shot"] = np.where(top2[:, 1] - top2[:, 0] > 0.02, sims.argmax(axis=1), ABSTAIN)

@labeling_function()
def zero_shot(x):
    return x.zero_shot
lfs.append(zero_shot)

applier = PandasLFApplier(lfs)
L_train, L_test = applier.apply(train, progress_bar=False), applier.apply(test, progress_bar=False)   # 文書数 × 関数の数の行列
# 説明のために学習データ全件の正解で評価している。本来は train の正解は使えないので、少量の人手のラベル付きの開発データで評価する
# Coverage: 棄権しなかった割合、Overlaps: ほかの関数と重なった割合、Conflicts: ほかの関数と違うラベルを付けた割合
summary = LFAnalysis(L_train, lfs).lf_summary(Y=train["y"].values)
print(summary[["Coverage", "Overlaps", "Conflicts", "Emp. Acc."]].round(3))
print("どれか 1 つでも当たった割合:", round((L_train != ABSTAIN).any(axis=1).mean(), 3))

               Coverage  Overlaps  Conflicts  Emp. Acc.
sports_kw         0.233     0.183      0.044      0.906
coach             0.116     0.112      0.073      0.384
movie_kw          0.273     0.211      0.135      0.818
smax_kw           0.253     0.201      0.028      0.937
dokujo_kw         0.234     0.178      0.084      0.786
title_bracket     0.114     0.110      0.058      0.718
zero_shot         0.591     0.545      0.101      0.966
どれか 1 つでも当たった割合: 0.944


In [4]:
from snorkel.labeling.model import LabelModel, MajorityLabelVoter

# 方法 1: 多数決。同票なら棄権
mv = MajorityLabelVoter(cardinality=4)
# 方法 2: ラベルモデル。関数どうしの一致・不一致のパターンだけから各関数の正解率を推定し、重み付きでまとめる (正解ラベルは使わない)
label_model = LabelModel(cardinality=4, verbose=False)
label_model.fit(L_train, n_epochs=500, lr=0.01, seed=0, progress_bar=False)
print(pd.DataFrame({"推定した正解率": label_model.get_weights(), "実際の正解率": summary["Emp. Acc."]}).round(3))

y_train = train["y"].values
pred_mv = mv.predict(L_train, tie_break_policy="abstain")
pred_lm = label_model.predict(L_train, tie_break_policy="abstain")
both = (pred_mv != ABSTAIN) & (pred_lm != ABSTAIN)
for name, pred in [("多数決", pred_mv), ("ラベルモデル", pred_lm)]:
    c = pred != ABSTAIN
    print(f"{name:7}: 付与率 {c.mean():.3f}  付与した中の正解率 {(pred[c] == y_train[c]).mean():.3f}"
          f"  (両方が付与した {both.sum()} 件での正解率 {(pred[both] == y_train[both]).mean():.3f})")

               推定した正解率  実際の正解率
sports_kw        0.889   0.906
coach            0.429   0.384
movie_kw         0.840   0.818
smax_kw          0.935   0.937
dokujo_kw        0.832   0.786
title_bracket    0.890   0.718
zero_shot        1.000   0.966
多数決    : 付与率 0.874  付与した中の正解率 0.947  (両方が付与した 2148 件での正解率 0.947)
ラベルモデル : 付与率 0.944  付与した中の正解率 0.920  (両方が付与した 2148 件での正解率 0.948)


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(analyzer="char", ngram_range=(1, 3), sublinear_tf=True, min_df=2).fit(train["text"])
X_train, X_test = vec.transform(train["text"]), vec.transform(test["text"])

y_test = test["y"].values

def test_pred(mask, labels):   # このセルは Colab の CPU で 1〜2 分
    # 簡単のため、確率的なラベルではなく最も確からしいラベル (棄権しなかった文書だけ) で学習する
    return LogisticRegression(C=10, max_iter=1000).fit(X_train[mask], labels[mask]).predict(X_test)

pred_test_lm = label_model.predict(L_test, tie_break_policy="random")   # 参考: ラベル付け関数だけでテストを分類 (全関数が棄権した文書は当て推量)
hit = (L_test != ABSTAIN).any(axis=1)
print(f"ラベル付け関数 + ラベルモデルでそのまま分類: {(pred_test_lm == y_test).mean():.3f}"
      f"  (どれかの関数が当たった {hit.sum()} 件: {(pred_test_lm[hit] == y_test[hit]).mean():.3f} / 全関数が棄権した {(~hit).sum()} 件: {(pred_test_lm[~hit] == y_test[~hit]).mean():.3f})")
for name, pred in [("多数決", pred_mv), ("ラベルモデル", pred_lm)]:
    p = test_pred(pred != ABSTAIN, pred)
    print(f"{name}のラベルで学習 ({(pred != ABSTAIN).sum()} 件): {(p == y_test).mean():.3f}"
          f"  (全関数が棄権した {(~hit).sum()} 件: {(p[~hit] == y_test[~hit]).mean():.3f})")
print(f"人手のラベルで学習 (全 {len(train)} 件): {(test_pred(np.ones(len(train), bool), y_train) == y_test).mean():.3f}")
for n in [10, 25, 50]:   # 人手のラベルが各クラス n 件しかない場合。選ぶ文書を 5 通り変える
    accs = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        idx = np.concatenate([rng.choice(np.where(y_train == k)[0], n, replace=False) for k in range(4)])
        mask = np.zeros(len(train), bool); mask[idx] = True
        accs.append((test_pred(mask, y_train) == y_test).mean())
    print(f"人手のラベルで学習 (各クラス {n} 件): 平均 {np.mean(accs):.3f}  範囲 {min(accs):.3f}〜{max(accs):.3f}")
print("テスト件数:", len(test))

ラベル付け関数 + ラベルモデルでそのまま分類: 0.890  (どれかの関数が当たった 991 件: 0.929 / 全関数が棄権した 62 件: 0.258)
多数決のラベルで学習 (2148 件): 0.973  (全関数が棄権した 62 件: 0.919)
ラベルモデルのラベルで学習 (2319 件): 0.978  (全関数が棄権した 62 件: 0.935)
人手のラベルで学習 (全 2457 件): 0.988
人手のラベルで学習 (各クラス 10 件): 平均 0.952  範囲 0.937〜0.968
人手のラベルで学習 (各クラス 25 件): 平均 0.969  範囲 0.963〜0.974
人手のラベルで学習 (各クラス 50 件): 平均 0.978  範囲 0.973〜0.981
テスト件数: 1053


- ラベル付け関数の出力を、関数どうしの一致・不一致から推定した正解率で重み付けしてまとめ、そのラベルで分類器を学習する枠組みは、Ratner ら（2016, NeurIPS（当時 NIPS））の「データプログラミング」。snorkel の `LabelModel` は、Ratner ら (2019, AAAI) の行列補完による推定を実装しており、正解ラベルを見ずに各関数の正解率を推定する。原論文は確率的なラベルのまま分類器を学習する（ノイズを考慮した損失）が、ここでは簡単のため最も確からしいラベルで学習した。
- 推定した正解率は、わざと入れた質の悪い関数 `coach`（実際 0.384）を 0.429 と低く見積もり、重みを下げた。一方で `title_bracket`（実際 0.718）は 0.890 と高く見積もった。`zero_shot` の 1.000 は `get_weights()` が上限 1.0 で切った値で、実際は 0.966。推定はあくまで目安である。一般に、ラベルモデルは関数どうしが（正解のラベルを固定すれば）独立だと仮定しており、同じ手がかりを見る関数が何本もあると推定が狂いやすい。
- 多数決とラベルモデルは、両方がラベルを付けた 2,148 件では正解率がほぼ同じ（0.947 と 0.948）だった。ラベルモデルは多数決が同票で棄権した文書にもラベルを付けるので付与率が高い (0.874 → 0.944) が、その分だけ正解率は下がる (0.920)。ラベルモデルが常に多数決より良いわけではないので、両方を試す。
- ラベル付け関数で直接テストデータを分類すると正解率 0.890 で、全関数が棄権した 62 件 (5.9%) は当て推量になり 0.258 だった。そのラベルで学習した TF-IDF の分類器は 0.973（多数決）/ 0.978（ラベルモデル）で、全関数が棄権した 62 件でも 0.919 / 0.935 だった。分類器はキーワード以外の手がかりも学ぶので、関数が当たらない文書にも汎化する。これが最後に分類器を学習する理由。なお 0.973 と 0.978 の差は、テスト 1,053 件のうち 5 件ほどで、ばらつきの範囲である。
- 人手のラベル全 2,457 件で学習すると 0.988 で、弱教師ありより 0.010（10 件ほど）高い。人手のラベルが各クラス 10 件 / 25 件 / 50 件のときは、選ぶ文書を 5 通り変えた平均で 0.952 / 0.969 / 0.978 だった。このデータでは、弱教師ありのラベルは人手の各クラス 25〜50 件（計 100〜200 件）と同程度である。
- ここでは説明のために、学習データ全件の正解ラベルで Coverage や Emp. Acc. を計算したが、本来は使えない。実際には数百件の人手のラベル付きの開発データを作り、それを見ながら関数を足したり直したりする。この例では同じ程度の件数（各クラス 50 件）で直接学習しても 0.978 が出るので、少量の人手のラベルで十分に分類できる課題では、ルールを書く手間に見合わない。ラベル付けが高くつく課題や、データが大量に流れてくる課題で効く。
- ラベル付け関数で作ったラベルには誤りが残る。この章の「学習データの中の付け間違えたラベルを見つける (Confident Learning)」の方法で、そうした誤りを洗い出すこともできる。
- snorkel (Apache 2.0) は、README に開発チームが商用の Snorkel Flow に注力すると書かれており、最後のリリースは 0.10.0（2024 年 2 月）。今の Colab (Python 3.13) では `pip install` で入り、動いた。データの livedoor ニュースコーパスは CC BY-ND 2.1 JP（改変禁止）、埋め込みモデル `intfloat/multilingual-e5-small` は MIT ライセンス。